# 01 · Linear algebra over GF(2), by hand

Every degeneracy in this repository is one number: $\log_2\mathrm{GSD} = n - \mathrm{rank}_{\mathbb F_2}F$.
This notebook is about how that rank, and the kernel that comes with it, are computed when the
numbers are bits. It writes Gaussian elimination over $GF(2)$ slowly enough to check every step,
runs it on matrices small enough to follow, and ends by checking that the fast, bit-packed routines
in `gsd.linalg` give the same answers.

**The only rule you need.** $GF(2)=\{0,1\}$ with arithmetic mod 2: addition is XOR.

| $+$ | 0 | 1 | | $\times$ | 0 | 1 |
|---|---|---|---|---|---|---|
| **0** | 0 | 1 | | **0** | 0 | 0 |
| **1** | 1 | 0 | | **1** | 0 | 1 |

Three consequences drive everything:

1. $1+1=0$, so $-a=a$: *subtracting a row and adding it are the same operation*. No minus signs anywhere.
2. The only nonzero scalar is $1$, so there is no "divide the pivot row by the pivot" step.
3. Row operations are their own inverses: doing $R_i \mathrel{+}= R_j$ twice changes nothing.

What is left of textbook elimination: find a pivot, swap it up, XOR it into every other row with a 1
in that column.

In [1]:
import sys
sys.path.insert(0, "..")

import itertools
import numpy as np
import gsd

## 1. The readable algorithm

`rref` sweeps the columns left to right with a counter `r` = "next row to fill".

* `r` advances **only** when a pivot is found, so rows `0..r-1` carry exactly one pivot each and
  `pivots[i]` is the pivot column of row `i`.
* The elimination loop runs over **all** rows, not only those below the pivot. That is what makes the
  result *reduced* (zeros above the pivots too), and it is what lets the kernel be read off in one line.
* Any row with a 1 is as good a pivot as any other: there is no rounding error over a finite field.

In [2]:
def rref(M):
    """Reduced row echelon form over GF(2). Returns (R, pivot_columns)."""
    M = np.array(M, dtype=np.int64) % 2      # never mutate the caller's matrix
    rows, cols = M.shape
    pivots, r = [], 0
    for col in range(cols):
        hits = [rr for rr in range(r, rows) if M[rr, col] == 1]
        if not hits:
            continue                         # free column: leave it, do NOT advance r
        M[[r, hits[0]]] = M[[hits[0], r]]    # bring the pivot row up to row r
        for rr in range(rows):               # ALL rows: this is what makes it *reduced*
            if rr != r and M[rr, col] == 1:
                M[rr] ^= M[r]                # XOR = add = subtract
        pivots.append(col)
        r += 1
        if r == rows:
            break
    return M, pivots


def nullspace(A):
    """Basis of {t : A t = 0} over GF(2): one vector per free column."""
    R, pivots = rref(A)
    cols = R.shape[1]
    basis = []
    for f in [c for c in range(cols) if c not in pivots]:
        v = np.zeros(cols, dtype=np.int64)
        v[f] = 1                             # switch this free variable on, the others off
        for i, p in enumerate(pivots):       # i = row of R, p = that row's pivot column
            v[p] = R[i, f]                   # back-substitution; no minus sign since -1 = 1
        basis.append(v)
    return np.array(basis).reshape(-1, cols)

## 2. The smallest interesting case

$$A=\begin{pmatrix}1&1\\1&1\end{pmatrix}$$

is the matrix of the stencil $1+x$ on a ring of $L=2$ sites: column $k$ is the stencil stamped at
site $k$, and on two sites the shift wraps around. The system $At=0$ is $t_0+t_1=0$ written twice.
RREF discovers that redundancy mechanically: $R_1 \leftarrow R_1+R_0 = (0,0)$.

In [3]:
A = np.array([[1, 1], [1, 1]])
R, piv = rref(A)
print("RREF:\n", R, "\npivot columns:", piv, "  rank =", len(piv))
print("kernel basis:", nullspace(A).tolist())
print("A @ (1,1) mod 2 =", (A @ [1, 1]) % 2)
assert (A == gsd.mult_matrix(gsd.Poly("1 + x", dim=1), (2,))).all()   # it really is 1 + x on 2 sites

RREF:
 [[1 1]
 [0 0]] 
pivot columns: [0]   rank = 1
kernel basis: [[1, 1]]
A @ (1,1) mod 2 = [0 0]


One free column, so one kernel vector, $t=(1,1)$: "apply the term at *both* sites" multiplies to the
identity. That is one relation among the terms, one lost rank, one factor of 2 in the degeneracy.
Over $GF(2)$ a $d$-dimensional subspace is a finite set of $2^d$ vectors; here $\ker A=\{(0,0),(1,1)\}$.

## 3. A wide matrix, traced step by step

$$A=\begin{pmatrix}1&1&0&1\\0&1&1&0\\1&0&1&1\end{pmatrix},\qquad R_2=R_0+R_1 .$$

Three equations, two independent: expect rank 2 and a 2-dimensional kernel.

In [4]:
A = np.array([[1, 1, 0, 1], [0, 1, 1, 0], [1, 0, 1, 1]])
M, rows = A.copy(), A.shape[0]
r = 0
for col in range(A.shape[1]):
    hits = [rr for rr in range(r, rows) if M[rr, col]]
    if not hits:
        print(f"col {col}: no pivot in rows >= {r} -> free column")
        continue
    M[[r, hits[0]]] = M[[hits[0], r]]
    cleared = [rr for rr in range(rows) if rr != r and M[rr, col]]
    for rr in cleared:
        M[rr] ^= M[r]
    print(f"col {col}: pivot row {r}, XOR it into rows {cleared}\n{M}")
    r += 1

col 0: pivot row 0, XOR it into rows [2]
[[1 1 0 1]
 [0 1 1 0]
 [0 1 1 0]]
col 1: pivot row 1, XOR it into rows [0, 2]
[[1 0 1 1]
 [0 1 1 0]
 [0 0 0 0]]
col 2: no pivot in rows >= 2 -> free column
col 3: no pivot in rows >= 2 -> free column


At column 1 the pivot row is XORed into row 0, which sits *above* it: that is the "reduced" in RREF.
The dependent row has become an explicit row of zeros, which is the general pattern: rank = number of
surviving nonzero rows.

**Reading off the kernel.** In reduced form row $i$ has a 1 in its own pivot column $p_i$, zeros in
the other pivot columns, and whatever else in the free columns, so $Rt=0$ says
$t_{p_i}=\sum_{f\,\mathrm{free}}R_{if}\,t_f$. Switching on one free variable at a time gives one basis
vector per free column, independent by construction (each is the only one with a 1 in its own free slot).

In [5]:
K = nullspace(A)
print("kernel basis:\n", K)
print("A K^T mod 2 =\n", (A @ K.T) % 2)

kernel basis:
 [[1 1 1 0]
 [1 0 0 1]]
A K^T mod 2 =
 [[0 0]
 [0 0]
 [0 0]]


### From a basis to the whole kernel

Over $GF(2)$ a linear combination is a choice of subset, and the combination is the XOR of that
subset. So the $2^d$ subsets of the $d$ basis vectors list the kernel exactly once each: nothing is
missed (there are no other coefficients), nothing repeats (two subsets with equal XOR would make the
basis dependent). That bijection is why $\mathrm{GSD}=2^{\dim}$ counts *states*.

In [6]:
d = K.shape[0]
for mask in range(2 ** d):
    t = sum((K[j] for j in range(d) if mask >> j & 1), np.zeros(A.shape[1], dtype=np.int64)) % 2
    print(f"mask {mask:0{d}b}  t = {t}  A t = {(A @ t) % 2}")

mask 00  t = [0 0 0 0]  A t = [0 0 0]
mask 01  t = [1 1 1 0]  A t = [0 0 0]
mask 10  t = [1 0 0 1]  A t = [0 0 0]
mask 11  t = [0 1 1 1]  A t = [0 0 0]


$(1,1,1,0)+(1,0,0,1)=(0,1,1,1)$ looks like neither generator and is just as much a relation: the
product of two operator identities is an operator identity. Enumerate only to build intuition: the
Newman–Moore model on a $63\times63$ torus already has $d=62$.

## 4. All sixteen $2\times2$ matrices

Small enough to list completely. Only 6 are invertible: they form $GL_2(\mathbb F_2)\cong S_3$, of order
$(2^2-1)(2^2-2)=6$.

In [7]:
print(" A            rank  kernel basis")
for bits in itertools.product([0, 1], repeat=4):
    A2 = np.array(bits).reshape(2, 2)
    print(f" {A2.tolist()}   {len(rref(A2)[1])}     {nullspace(A2).tolist()}")

 A            rank  kernel basis
 [[0, 0], [0, 0]]   0     [[1, 0], [0, 1]]
 [[0, 0], [0, 1]]   1     [[1, 0]]
 [[0, 0], [1, 0]]   1     [[0, 1]]
 [[0, 0], [1, 1]]   1     [[1, 1]]
 [[0, 1], [0, 0]]   1     [[1, 0]]
 [[0, 1], [0, 1]]   1     [[1, 0]]
 [[0, 1], [1, 0]]   2     []
 [[0, 1], [1, 1]]   2     []
 [[1, 0], [0, 0]]   1     [[0, 1]]
 [[1, 0], [0, 1]]   2     []
 [[1, 0], [1, 0]]   1     [[0, 1]]
 [[1, 0], [1, 1]]   2     []
 [[1, 1], [0, 0]]   1     [[1, 1]]
 [[1, 1], [0, 1]]   2     []
 [[1, 1], [1, 0]]   2     []
 [[1, 1], [1, 1]]   1     [[1, 1]]


## 5. Where the free column is not last

$M=\begin{pmatrix}1&1&0\\0&0&1\end{pmatrix}$ is already reduced, with pivots $\{0,2\}$ and the free
column **between** them. Row $i$ must be paired with the $i$-th *pivot column*, not with column $i$ —
exactly what `enumerate(pivots)` does. The kernel is $(1,1,0)$.

In [8]:
print(nullspace([[1, 1, 0], [0, 0, 1]]))

[[1 1 0]]


## 6. The ring $\mathbb F_2[x]/(x^L-1)$: the stencil $1+x$ on any ring

Multiplication by $1+x$ on $L$ sites is the circulant matrix with $C_{ik}=1$ iff $i=k$ or $i=k+1$.
Row $i$ of $Ct=0$ says $t_i=t_{i-1}$, so $t$ is constant: the kernel is always $\{\mathbf 0,\mathbf 1\}$.
RREF sees $L-1$ pivots and one free column, whose back-substitution is the all-ones vector.

In [9]:
for L in (4, 5, 9):
    C = gsd.mult_matrix(gsd.Poly("1 + x", dim=1), (L,))
    print(f"L={L}: rank {len(rref(C)[1])}, kernel {nullspace(C).tolist()}")
print(rref(gsd.mult_matrix(gsd.Poly("1 + x", dim=1), (4,)))[0])

L=4: rank 3, kernel [[1, 1, 1, 1]]
L=5: rank 4, kernel [[1, 1, 1, 1, 1]]
L=9: rank 8, kernel [[1, 1, 1, 1, 1, 1, 1, 1, 1]]
[[1 0 0 1]
 [0 1 0 1]
 [0 0 1 1]
 [0 0 0 0]]


## 7. Beyond $p=2$

For qudits ($\mathbb Z_p$, $p$ prime) two things change, and nothing else:

1. **Normalise the pivot** by its inverse $a^{-1}=a^{p-2}\bmod p$ (Fermat).
2. **Elimination subtracts**, and back-substitution keeps its sign: $v_{p_i}=-R_{if}$.

`gsd.rref(A, p)`, `gsd.rank(A, p)` and `gsd.nullspace(A, p)` implement both cases. For $p=2$ they pack 64
matrix entries into each machine word, so one XOR eliminates 64 entries at once. The last cell checks
them against the readable versions above on random matrices.

In [10]:
rng = np.random.default_rng(1)
for _ in range(200):
    A = rng.integers(0, 2, size=rng.integers(1, 30, size=2))
    R, piv = rref(A)
    R2, piv2 = gsd.rref(A)
    assert piv == piv2 and (R == R2).all()
    assert sorted(map(tuple, nullspace(A))) == sorted(map(tuple, gsd.nullspace(A)))
print("readable and fast routines agree on 200 random matrices")

A = np.array([[1, 2, 0], [2, 1, 1]])
print("over GF(3):  rank", gsd.rank(A, 3), "  kernel", gsd.nullspace(A, 3).tolist(),
      "  check:", (A @ gsd.nullspace(A, 3).T % 3).ravel())

readable and fast routines agree on 200 random matrices
over GF(3):  rank 2   kernel [[1, 1, 0]]   check: [0 0]
